In [9]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("spam_train_cleaned.csv")
test_df = pd.read_csv("spam_test_cleaned.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())

Train shape: (31311, 16)
Test shape : (7828, 16)

Train columns:
['label', 'urls', 'hour', 'combined_text', 'capital_letter_count', 'capital_ratio', 'exclamation_count', 'question_count', 'special_char_count', 'day_of_week_Friday', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']


,label,urls,hour,combined_text,capital_letter_count,capital_ratio,exclamation_count,question_count,special_char_count,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,0,1,2,volunteers are needed for alumni phonothon 200...,37,0.055306,0,0,2,0,0,0,0,0,0,1
1,0,0,0,re opensuse opensuse and faxes on sunday 10 fe...,85,0.048935,0,2,42,0,0,0,0,0,0,1
2,0,1,2,re r matching a period in grep on 08 05 2008 0...,69,0.045128,0,4,114,0,0,0,0,0,0,1
3,1,1,17,fast and safe male enhancement huge love gun i...,11,0.034700,3,0,0,0,0,0,0,1,0,0
4,0,1,3,re python dev documentation reorganization was...,44,0.026113,0,0,94,0,0,0,0,0,0,1


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

# =========================
# X and y
# =========================

X = train_df.drop(columns=["label"])
y = train_df["label"]

X_test_raw = test_df.drop(columns=["label"])
y_test = test_df["label"]


# =========================
# Train / Validation split
# =========================

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# =========================
# Text + numerical columns
# =========================

TEXT_COL = "combined_text"

NUMERIC_COLS = [
    col for col in X.columns
    if col != TEXT_COL
]


# =========================
# TF-IDF
# =========================

tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_text_train = tfidf.fit_transform(X_train_raw[TEXT_COL])
X_text_val = tfidf.transform(X_val_raw[TEXT_COL])
X_text_test = tfidf.transform(X_test_raw[TEXT_COL])


# =========================
# Scaling
# =========================

scaler = StandardScaler()

X_num_train = scaler.fit_transform(X_train_raw[NUMERIC_COLS])
X_num_val = scaler.transform(X_val_raw[NUMERIC_COLS])
X_num_test = scaler.transform(X_test_raw[NUMERIC_COLS])


# =========================
# Combine ALL features
# =========================

X_train = hstack([
    X_text_train,
    csr_matrix(X_num_train)
])

X_val = hstack([
    X_text_val,
    csr_matrix(X_num_val)
])

X_test = hstack([
    X_text_test,
    csr_matrix(X_num_test)
])


print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (25048, 314)
Validation: (6263, 314)
Test: (7828, 314)


In [11]:
#Default SVM no hyperpramters
from sklearn.svm import SVC

svm_default = SVC()

svm_default.fit(X_train, y_train)

print("Default SVM trained!")

Default SVM trained!


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_train_pred = svm_default.predict(X_train)
y_val_pred = svm_default.predict(X_val)

print("========== DEFAULT SVM ==========")

print("\nTRAINING PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_train, y_train_pred), 4))
print("Precision:", round(precision_score(y_train, y_train_pred), 4))
print("Recall   :", round(recall_score(y_train, y_train_pred), 4))
print("F1 Score :", round(f1_score(y_train, y_train_pred), 4))

print("\nVALIDATION PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred), 4))
print("Precision:", round(precision_score(y_val, y_val_pred), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred), 4))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, y_val_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_val, y_val_pred))

========== DEFAULT SVM ==========

TRAINING PERFORMANCE
Accuracy : 0.986
Precision: 0.9835
Recall   : 0.9916
F1 Score : 0.9875

VALIDATION PERFORMANCE
Accuracy : 0.9823
Precision: 0.9801
Recall   : 0.9883
F1 Score : 0.9842

CONFUSION MATRIX
[[2700   70]
 [  41 3452]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      2770
           1       0.98      0.99      0.98      3493

    accuracy                           0.98      6263
   macro avg       0.98      0.98      0.98      6263
weighted avg       0.98      0.98      0.98      6263

